<a href="https://colab.research.google.com/github/LefieTsimplekas/thesis-bikeshare-synthesis/blob/main/notebooks/colab_train_tabddpm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Εκπαίδευση TabDDPM στο Google Colab

**Πριν ξεκινήσεις:** `Runtime → Change runtime type → T4 GPU`.

Στο Google Drive χρειάζεται ο φάκελος `MyDrive/thesis/model_input/` με:
- τα 9 αρχεία του `data/model_input/` (`A_train.parquet` … `D_val.parquet`, `london_test.parquet`)
- το `artifacts/model_input/schema.json`

Τα αποτελέσματα γράφονται στο `MyDrive/thesis/outputs/`, ώστε να μη χάνονται όταν κλείσει η συνεδρία.

## 1. Έλεγχος GPU

In [1]:
!nvidia-smi

Tue Nov 10 07:24:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Εγκατάσταση (μία φορά ανά συνεδρία, ~3–5 λεπτά)

Το Colab τρέχει πλέον Python 3.13, για την οποία δεν υπάρχουν τα torch 2.2.2 και numpy 1.26.4 που απαιτεί το synthcity. Γι' αυτό φτιάχνεται ένα ξεχωριστό περιβάλλον **Python 3.11** με το `uv`, με τις ίδιες εκδόσεις όπως το `requirements.txt`. Τα scripts τρέχουν με αυτό (`/content/py311/bin/python`). **Δεν χρειάζεται Restart session.**

In [2]:
!pip install -q uv
!uv venv /content/py311 --python 3.11 -q
!uv pip install -q --python /content/py311/bin/python torch==2.2.2 numpy==1.26.4 pandas==2.3.3 pyarrow==24.0.0 scipy==1.17.1 scikit-learn==1.8.0 xgboost==2.1.4 synthcity==0.2.12 opacus==1.4.0

## 3. Drive, κώδικας και φάκελοι

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
REPO = '/content/thesis-bikeshare-synthesis'
DRIVE = '/content/drive/MyDrive/thesis'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/LefieTsimplekas/thesis-bikeshare-synthesis', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
os.chdir(REPO)

# Είσοδοι από το Drive
os.makedirs('data', exist_ok=True); os.makedirs('artifacts/model_input', exist_ok=True)
if not os.path.exists('data/model_input'):
    os.symlink(f'{DRIVE}/model_input', 'data/model_input')
if not os.path.exists('artifacts/model_input/schema.json'):
    os.symlink(f'{DRIVE}/model_input/schema.json', 'artifacts/model_input/schema.json')

# Έξοδοι στο Drive
for local, remote in [('data/synthetic', 'synthetic'), ('models', 'models'), ('artifacts/training', 'training')]:
    os.makedirs(f'{DRIVE}/outputs/{remote}', exist_ok=True)
    if not os.path.exists(local):
        os.symlink(f'{DRIVE}/outputs/{remote}', local)

print(sorted(os.listdir('data/model_input')))

PY = '/content/py311/bin/python'
subprocess.run([PY, '-c', "import sys, torch, numpy; print('python', sys.version.split()[0], '| torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| numpy', numpy.__version__)"], check=True)

Mounted at /content/drive
['manifest.json']


CompletedProcess(args=['/content/py311/bin/python', '-c', "import sys, torch, numpy; print('python', sys.version.split()[0], '| torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| numpy', numpy.__version__)"], returncode=0)

In [4]:
import os
os.chdir('/content/thesis-bikeshare-synthesis')
DRIVE = '/content/drive/MyDrive/thesis'
os.makedirs('data/model_input', exist_ok=True); os.makedirs('artifacts/model_input', exist_ok=True)
for f in os.listdir(f'{DRIVE}/model_input'):
    dst = 'artifacts/model_input/schema.json' if f == 'schema.json' else f'data/model_input/{f}'
    if not os.path.exists(dst):
        os.symlink(f'{DRIVE}/model_input/{f}', dst)
print(sorted(os.listdir('data/model_input')))

['A_train.parquet', 'A_val.parquet', 'B_train.parquet', 'B_val.parquet', 'C_train.parquet', 'C_val.parquet', 'D_train.parquet', 'D_val.parquet', 'london_test.parquet', 'manifest.json']


## 4. Πιλοτική εκτέλεση (σενάριο D, 50.000 γραμμές, 5.000 βήματα)

Ελέγχει ότι όλα δουλεύουν και μετρά την ταχύτητα. Στο «Quick check» στο τέλος, τα ποσοστά και οι διάμεσοι του συνθετικού πρέπει να είναι κοντά σε αυτούς του train.

In [ ]:
!/content/py311/bin/python src/07_train_tabddpm.py --scenario D --pilot-rows 50000 --steps 5000 --gen-rows 20000 2>&1 | grep -v 'Epoch'

[transformers] Disabling PyTorch because PyTorch >= 2.6 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[2026-11-09T21:56:03.232780+0000][5575][CRITICAL] Error importing TabularGoggle: No module named 'dgl'
[2026-11-09T21:56:03.235387+0000][5575][CRITICAL] module disabled: /content/py311/lib/python3.11/site-packages/synthcity/plugins/generic/plugin_goggle.py

===== TabDDPM | scenario D =====
Rows 50,000 | batch 4096 | 13 steps/epoch | 385 epochs ≈ 5,005 steps | device cuda
Training time 6.2 min (13.5 steps/s)
Generation of 20,000 rows: 0.9 min
Quick check (train vs synthetic):
  train     e-bike  18.2% | round trip  2.5% | median duration 13.34 min | median distance  2.66 km
  synthetic e-bike  18.1% | round trip  2.3% | median duration 13.52 min | median distance  2.71 km
Saved data/synthetic/tabddpm_D_pilot.parquet


## 5. Πλήρης εκπαίδευση (ένα σενάριο ανά κελί)

Τρέξε τα κελιά ένα ένα. Αν η συνεδρία κλείσει, ό,τι έχει τελειώσει είναι ήδη στο Drive· ξανατρέχεις τα κελιά 2 και 3 και μετά το επόμενο σενάριο.

In [5]:
!/content/py311/bin/python src/07_train_tabddpm.py --scenario D 2>&1 | grep -v 'Epoch'

[transformers] Disabling PyTorch because PyTorch >= 2.6 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[KeOps] Compiling cuda jit compiler engine ... OK
[pyKeOps] Compiling nvrtc binder for python ... OK
[2026-11-10T07:28:25.574932+0000][1626][CRITICAL] Error importing TabularGoggle: No module named 'dgl'
[2026-11-10T07:28:25.579960+0000][1626][CRITICAL] module disabled: /content/py311/lib/python3.11/site-packages/synthcity/plugins/generic/plugin_goggle.py

===== TabDDPM | scenario D =====
Rows 720,809 | batch 4096 | 176 steps/epoch | 114 epochs ≈ 20,064 steps | device cuda
Training time 27.6 min (12.1 steps/s)
Generation of 200,000 rows: 8.4 min
Quick check (train vs synthetic):
  train     e-bike  18.1% | round trip  2.5% | median duration 13.27 min | median distance  2.65 km
  synthetic e-bike  18.1% | round trip  2.4% | median duration 13.41 min | median distance  2.

In [6]:
!/content/py311/bin/python src/07_train_tabddpm.py --scenario A 2>&1 | grep -v 'Epoch'

[transformers] Disabling PyTorch because PyTorch >= 2.6 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[2026-11-10T08:07:29.252558+0000][12165][CRITICAL] Error importing TabularGoggle: No module named 'dgl'
[2026-11-10T08:07:29.255241+0000][12165][CRITICAL] module disabled: /content/py311/lib/python3.11/site-packages/synthcity/plugins/generic/plugin_goggle.py

===== TabDDPM | scenario A =====
Rows 720,809 | batch 4096 | 176 steps/epoch | 114 epochs ≈ 20,064 steps | device cuda
Training time 26.9 min (12.4 steps/s)
Generation of 200,000 rows: 8.5 min
Quick check (train vs synthetic):
  train     e-bike  46.9% | round trip  4.1% | median duration  9.87 min | median distance  2.09 km
  synthetic e-bike  47.1% | round trip  4.1% | median duration  9.91 min | median distance  2.09 km
Saved data/synthetic/tabddpm_A.parquet


In [7]:
!/content/py311/bin/python src/07_train_tabddpm.py --scenario B 2>&1 | grep -v 'Epoch'

[transformers] Disabling PyTorch because PyTorch >= 2.6 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[2026-11-10T08:46:11.822803+0000][22447][CRITICAL] Error importing TabularGoggle: No module named 'dgl'
[2026-11-10T08:46:11.825451+0000][22447][CRITICAL] module disabled: /content/py311/lib/python3.11/site-packages/synthcity/plugins/generic/plugin_goggle.py

===== TabDDPM | scenario B =====
Rows 720,809 | batch 4096 | 176 steps/epoch | 114 epochs ≈ 20,064 steps | device cuda
Training time 26.8 min (12.5 steps/s)
Generation of 200,000 rows: 8.3 min
Quick check (train vs synthetic):
  train     e-bike  70.4% | round trip  1.9% | median duration  8.78 min | median distance  1.96 km
  synthetic e-bike  70.5% | round trip  1.8% | median duration  8.92 min | median distance  2.00 km
Saved data/synthetic/tabddpm_B.parquet


In [8]:
!/content/py311/bin/python src/07_train_tabddpm.py --scenario C 2>&1 | grep -v 'Epoch'

[transformers] Disabling PyTorch because PyTorch >= 2.6 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[2026-11-10T09:27:41.780533+0000][33464][CRITICAL] Error importing TabularGoggle: No module named 'dgl'
[2026-11-10T09:27:41.783248+0000][33464][CRITICAL] module disabled: /content/py311/lib/python3.11/site-packages/synthcity/plugins/generic/plugin_goggle.py

===== TabDDPM | scenario C =====
Rows 720,809 | batch 4096 | 176 steps/epoch | 114 epochs ≈ 20,064 steps | device cuda
Training time 27.3 min (12.2 steps/s)
Generation of 200,000 rows: 8.7 min
Quick check (train vs synthetic):
  train     e-bike  58.6% | round trip  3.0% | median duration  9.30 min | median distance  2.03 km
  synthetic e-bike  58.7% | round trip  3.1% | median duration  9.32 min | median distance  2.05 km
Saved data/synthetic/tabddpm_C.parquet
